Perfect ✅
Let’s build a **real, production-ready LangGraph + MCP project**.

We’ll design something realistic but clean:

---

# 🚀 Project: AI Dev Assistant (LangGraph + MCP)

An AI coding assistant that can:

* 📂 Read & write files
* 🧪 Run tests
* 📊 Query a database
* 📧 Send email summaries
* 🧠 Use structured ReAct reasoning
* 🔌 Dynamically discover tools via MCP

Production-ready. Modular. Clean.

---

# 🏗 Architecture Overview

![Image](https://miro.medium.com/v2/resize%3Afit%3A2000/1%2AzN71LjNRbJqD6SyNaxKr2g.png)

![Image](https://miro.medium.com/0%2Afmk9z4CULIH1fd9w.png)

![Image](https://miro.medium.com/v2/resize%3Afit%3A1400/0%2AJSkTXRnI1UozuDoC.jpg)

![Image](https://miro.medium.com/v2/resize%3Afit%3A1400/0%2AW-bBkmKrmXyTKX3U.png)

```text
User
  ↓
LangGraph Agent
  ↓
MCP Client Layer
  ↓
MCP Servers
   ├── filesystem server
   ├── database server
   ├── email server
   └── test runner server
```

LangGraph = Brain
MCP = Tool Infrastructure

---

# 📁 Production-Ready Folder Structure

Simple. Clean. Scalable.

```
ai-dev-assistant/
│
├── app/
│   ├── main.py                 # Entry point
│   ├── config.py               # Settings (env, model config)
│   │
│   ├── graph/
│   │   ├── state.py            # LangGraph state definition
│   │   ├── builder.py          # Graph construction
│   │   ├── nodes/
│   │   │   ├── llm_node.py
│   │   │   ├── tool_node.py
│   │   │   └── router_node.py
│   │
│   ├── mcp/
│   │   ├── client.py           # MCP client wrapper
│   │   └── tool_adapter.py     # Convert MCP tools → LLM schema
│
├── servers/
│   ├── filesystem_server.py
│   ├── database_server.py
│   ├── email_server.py
│   └── test_server.py
│
├── tests/
│   └── test_graph.py
│
├── .env
├── requirements.txt
└── README.md
```

Minimal but scalable.

---

# 🧠 app/graph/state.py

```python
from typing import TypedDict, List, Any

class AgentState(TypedDict):
    messages: List[dict]
    tool_calls: List[dict]
    tool_results: List[Any]
    final_response: str
```

State-driven execution.

---

# 🔌 app/mcp/client.py

Reusable MCP client wrapper.

```python
import asyncio
from mcp.client.session import ClientSession
from mcp.client.stdio import stdio_client

class MCPClient:

    def __init__(self, command: str):
        self.command = command
        self.session = None

    async def connect(self):
        read, write = await stdio_client(self.command)
        self.session = await ClientSession(read, write).__aenter__()

    async def list_tools(self):
        return await self.session.list_tools()

    async def call_tool(self, name: str, args: dict):
        return await self.session.call_tool(name, args)
```

Production tip:
Connect once at startup. Reuse session.

---

# 🔄 app/mcp/tool_adapter.py

Converts MCP schema → LLM schema.

```python
def convert_mcp_tools_to_llm(tools):

    llm_tools = []

    for tool in tools:
        llm_tools.append({
            "name": tool.name,
            "description": tool.description,
            "parameters": tool.inputSchema
        })

    return llm_tools
```

Keeps LLM decoupled from MCP.

---

# 🤖 app/graph/nodes/llm_node.py

```python
def llm_node(state, llm):

    response = llm.invoke(state["messages"])

    return {
        "messages": state["messages"] + [response],
        "tool_calls": getattr(response, "tool_calls", [])
    }
```

---

# 🛠 app/graph/nodes/tool_node.py

```python
async def tool_node(state, mcp_client):

    results = []

    for call in state["tool_calls"]:
        result = await mcp_client.call_tool(
            call["name"],
            call["args"]
        )
        results.append(result)

    return {
        "tool_results": results
    }
```

---

# 🔀 app/graph/builder.py

```python
from langgraph.graph import StateGraph, END

def build_graph(llm, mcp_client, state_cls):

    builder = StateGraph(state_cls)

    builder.add_node("llm", lambda s: llm_node(s, llm))
    builder.add_node("tools", lambda s: tool_node(s, mcp_client))

    builder.add_edge("llm", "tools")
    builder.add_edge("tools", "llm")

    builder.set_entry_point("llm")

    return builder.compile()
```

ReAct loop.

---

# 🚀 app/main.py

```python
import asyncio
from graph.state import AgentState
from graph.builder import build_graph
from mcp.client import MCPClient
from mcp.tool_adapter import convert_mcp_tools_to_llm

async def main():

    mcp = MCPClient("python servers/filesystem_server.py")
    await mcp.connect()

    tools = await mcp.list_tools()
    llm_tools = convert_mcp_tools_to_llm(tools)

    llm = create_llm_with_tools(llm_tools)

    graph = build_graph(llm, mcp, AgentState)

    result = await graph.ainvoke({
        "messages": [{"role": "user", "content": "Read main.py"}]
    })

    print(result)

asyncio.run(main())
```

---

# 🖥 Example MCP Server (servers/filesystem_server.py)

```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("filesystem-server")

@mcp.tool()
def read_file(path: str) -> str:
    """Read file content"""
    with open(path, "r") as f:
        return f.read()

if __name__ == "__main__":
    mcp.run()
```

Add more servers for:

* database
* email
* test runner

Each is independent microservice.

---

# 🔐 Production Best Practices

### 1️⃣ Separate MCP servers by domain

Filesystem ≠ Database ≠ Email

### 2️⃣ Never mix business logic into graph

Graph = orchestration only

### 3️⃣ Add logging middleware

Track tool calls

### 4️⃣ Add timeout handling

MCP tools may hang

### 5️⃣ Add auth layer (if remote servers)

---

# 🏢 Production Deployment Structure

In real deployment:

```
Docker containers:
  - agent container
  - filesystem MCP container
  - database MCP container
  - email MCP container
```

Agent scales independently.

---

# 🎯 What Makes This Production Ready?

✔ Clear separation of concerns
✔ Reusable MCP client layer
✔ Modular graph nodes
✔ Tool discovery dynamic
✔ Microservice architecture
✔ Easy scaling
✔ Easy testing

---

# 🧠 Final Architecture Mental Model

```text
LangGraph → controls thinking
MCP → controls doing
Servers → execute real work
```

This is enterprise-grade AI architecture.

---

# 🏗 Analogy — Modern Cloud App

Think of:

Frontend → LangGraph
API Gateway → MCP Client
Microservices → MCP Servers
Databases → Resources

Clean layered system.

---

# If You Want Next:

I can now give you:

1. 🔥 Multi-agent supervisor architecture (advanced)
2. 🧪 Full testable production template (with logging, retries)
3. 🐳 Dockerized deployment setup
4. 🏭 Enterprise-scale architecture diagram
5. 🧠 Advanced memory + MCP integration

Tell me what level you want next.


You’re thinking in the right direction. Let’s convert your intuition into a **real production-grade agent architecture**.

I’ll explain this in 5 parts:

1. Overall flow
2. Tools list and responsibilities
3. How the agent “thinks” and makes decisions
4. Scheduler vs WebSocket (LinkedIn job fetching strategy)
5. Complete high-level architecture design

And I’ll use a generalized analogy at the end.

---

# 1️⃣ End-to-End Flow (Real Production Flow)

![Image](https://images.ctfassets.net/lzny33ho1g45/1v6L0TqwBK5kXnjVt7tqLa/3d0a958c05a11061e6cd723700eb9aab/AI-automation-components.jpg)

![Image](https://storage.googleapis.com/typhoon-blog-assets/images/agent_loop.png)

![Image](https://miro.medium.com/v2/resize%3Afit%3A1400/1%2A3Gm--iPcO7zuBMvMzvt2mw.png)

![Image](https://svg.template.creately.com/geqyowvc6)

```text
Step 0: Scheduler triggers agent daily

Step 1: Agent fetches jobs using Job Fetch Tool

Step 2: Agent filters relevant jobs

Step 3: Agent reads job description

Step 4: Agent uses Resume Tool to customize resume

Step 5: Agent decides how to apply:
        Option A: LinkedIn Easy Apply
        Option B: Redirect → Company Portal

Step 6: Agent uses Apply Tool

Step 7: Agent stores application result in Tracker Tool

Step 8: Dashboard reads database and shows analytics
```

---

# 2️⃣ Core Tools (Your Agent Toolkit)

Think of agent as brain and tools as hands.

You need 5 core tools:

---

## Tool 1: Job Fetch Tool

Purpose:
Fetch jobs from LinkedIn, Wellfound, company portals.

Input:

```python
{
  "role": "Data Scientist",
  "location": "Remote",
  "experience": "2 years",
  "companies": ["Google", "Meta"]
}
```

Output:

```python
[
  {
    "title": "Data Scientist",
    "company": "Google",
    "apply_url": "...",
    "description": "...",
    "portal": "linkedin"
  }
]
```

This tool does:

* scraping OR
* API fetching OR
* browser automation

---

## Tool 2: Resume Optimizer Tool (Gemini)

Purpose:
Customize resume per job.

Input:

```python
{
  "base_resume": "...",
  "job_description": "..."
}
```

Output:

```python
optimized_resume_text
```

This improves ATS score.

---

## Tool 3: Apply Tool

Handles two cases:

---

### Case A: LinkedIn Easy Apply

Agent fills:

```text
Name
Email
Resume upload
Questions
Submit
```

---

### Case B: Company Portal Apply

Agent detects portal type:

```text
Greenhouse
Lever
Workday
Custom portal
```

Then fills form.

---

## Tool 4: Tracker Tool

Stores:

```python
{
  company,
  role,
  status,
  date,
  portal
}
```

Database = SQLite/Postgres

---

## Tool 5: Dashboard Tool

Reads DB.

Shows:

* total applications
* per company
* per role
* success rate

---

# 3️⃣ How Agent Thinks (Decision Logic)

Agent reasoning loop:

```text
Goal: Apply to relevant jobs

Step 1: Fetch jobs

Step 2: For each job:

   Is job relevant?
      yes → continue
      no → skip

Step 3: Customize resume

Step 4: Determine apply method:

   if easy_apply:
       use linkedin apply tool

   else:
       use browser apply tool

Step 5: Track result

Step 6: Continue loop
```

This is classic agent loop:

```text
Observe → Think → Act → Observe → Think → Act
```

---

# 4️⃣ LinkedIn Fetch Strategy (IMPORTANT)

You asked:

> WebSocket or Scheduler?

Answer:

You MUST use Scheduler.

NOT WebSocket.

---

## Why WebSocket won’t work

LinkedIn does NOT push jobs to you.

WebSocket works only when server supports push.

LinkedIn does not expose such API.

---

## Correct Approach = Scheduler

Example:

```text
Every day at 9 AM

Scheduler triggers:

fetch_linkedin_jobs()
```

Tools:

* cron
* APScheduler
* Celery
* Airflow

---

Example scheduler:

```python
import schedule
import time

def job():
    agent.run()

schedule.every().day.at("09:00").do(job)

while True:
    schedule.run_pending()
    time.sleep(60)
```

---

# 5️⃣ Complete High-Level Architecture

![Image](https://www.researchgate.net/publication/330075047/figure/fig2/AS%3A710336722243585%401546368880307/The-system-architecture-of-scheduler-based-microservice.jpg)

![Image](https://cdn.prod.website-files.com/5fff1b18d19a56869649c806/6802998d8fb9294797720dcf_architecture%20diagram%20llm%20ai%20agent%20sensitive%20data.webp)

![Image](https://www.slideteam.net/media/catalog/product/cache/1280x720/f/r/frontend_and_backend_database_architecture_diagram_slide01.jpg)

![Image](https://goldenowl-asia-assets-production.s3.ap-southeast-1.amazonaws.com/uploads/2025-05-12T08%3A02%3A01.903Z_hxxeuquwowgp4mzoknp1%20%281%29.webp)

```text
                Scheduler
                    ↓
                Agent Core
                    ↓
    ┌───────────────┼───────────────┐
    ↓               ↓               ↓
Job Tool      Resume Tool      Apply Tool
    ↓               ↓               ↓
LinkedIn      Gemini API       Browser
Wellfound                       Automation
Company portals

                    ↓
               Tracker Tool
                    ↓
                Database
                    ↓
                Dashboard
```

---

# 6️⃣ Internal Agent Structure (Real Code Architecture)

```python
class JobAgent:

    def run(self):

        jobs = job_fetch_tool()

        for job in jobs:

            if not self.is_relevant(job):
                continue

            resume = resume_tool(job)

            apply_result = apply_tool(job, resume)

            tracker_tool(apply_result)
```

---

# 7️⃣ Portal Detection Logic

When agent clicks Apply:

Agent checks URL:

```text
linkedin.com → linkedin apply

greenhouse.io → greenhouse apply

lever.co → lever apply

workday.com → workday apply
```

Each portal has different handler.

---

# 8️⃣ Realistic Production Flow Example

Example run:

```text
Scheduler starts agent

Agent fetches 100 jobs

Agent filters → 20 relevant

Agent customizes 20 resumes

Agent applies to 20 jobs

Agent stores results

Dashboard updates
```

---

# 9️⃣ Recommended Production Stack

```text
LLM → Gemini

Browser automation → Playwright

Scheduler → APScheduler

Backend → Python

Database → PostgreSQL

Dashboard → Streamlit

Scraping → Playwright + BeautifulSoup
```

---

# 10️⃣ How Real Companies Build Similar Systems

Companies build systems like this for:

* internal hiring automation
* candidate matching
* resume optimization

Structure is exactly same.

---

# Final Analogy — Personal Job Assistant 👔

Imagine you hired a human assistant.

They do:

Morning:
Check LinkedIn and job sites

For each job:
Modify your resume

Go to company website

Fill application

Record in spreadsheet

Show you dashboard

Your AI agent does same.

Scheduler = alarm clock
Agent = assistant brain
Tools = assistant hands
Database = spreadsheet
Dashboard = report

---

# If you want next, I’ll give:

Production-grade tool definitions like real agent:

```python
fetch_jobs_tool()
optimize_resume_tool()
apply_job_tool()
track_job_tool()
```

and full agent implementation.

Just say: **build full agent code**


Excellent question 👌 — this is the **core engineering challenge**.

You're basically asking:

> How does the Apply Agent actually fill forms?
> Does it type like a human?
> Does it inject data?
> How does it know which field is what?

Let’s break this down properly.

---

# 🧠 Big Picture

There are only **3 real ways** an agent can fill job applications:

1. DOM-based field detection (most common, safest)
2. Browser automation (human-like typing)
3. Direct API submission (rare, only when available)

In real-world production → we use **Browser Automation + Intelligent Field Mapping**

---

# 🖥️ How It Actually Works (Under the Hood)

![Image](https://axiom.ai/assets/img/form-filling-design-pattern-for-bot.4f024e28.png)

![Image](https://miro.medium.com/v2/resize%3Afit%3A1400/1%2A8CVZIQOm1Wrb4gSm2UMnvw.png)

![Image](https://www.researchgate.net/publication/357289431/figure/fig1/AS%3A1104342824300544%401640307255477/The-system-components-diagram-for-detecting-phishing-URLs-in-a-real-time-environment.png)

![Image](https://www.researchgate.net/publication/375131389/figure/fig1/AS%3A11431281202405636%401698843118881/AI-fields-illustrations.ppm)

The agent does this:

```text
1. Open job application page
2. Inspect HTML DOM
3. Identify input fields
4. Match fields with your profile data
5. Fill fields
6. Upload resume
7. Click submit
```

---

# 🔎 Step 1 — Detect Form Fields

Every input field in HTML looks like:

```html
<input type="text" name="firstName" />
<input type="email" name="email" />
<textarea name="coverLetter"></textarea>
```

The browser automation tool (Playwright) can read this.

Example:

```python
page.query_selector_all("input")
```

The agent collects:

* field name
* id
* placeholder
* label text
* type (text, email, file, checkbox)

---

# 🧠 Step 2 — Field Mapping Intelligence

The agent must answer:

> This input field is asking for what?

It uses heuristics:

If:

```text
name contains "first"
label contains "First Name"
placeholder contains "First Name"
```

→ map to `profile.first_name`

So internally:

```python
PROFILE = {
    "first_name": "Rahul",
    "last_name": "Sharma",
    "email": "rahul@email.com",
    "phone": "9876543210",
    "linkedin": "linkedin.com/in/rahul"
}
```

Then:

```python
page.fill("input[name='firstName']", PROFILE["first_name"])
```

---

# ⌨️ Step 3 — How It Types

Playwright can:

### Option A — Instant fill (fast)

```python
page.fill(selector, value)
```

### Option B — Human-like typing

```python
page.type(selector, value, delay=50)
```

The second is safer (less bot-like).

---

# 📎 Step 4 — Resume Upload

File upload is easy:

```python
page.set_input_files("input[type='file']", "resume.pdf")
```

---

# 🧾 Step 5 — Handling Dropdowns

Example:

```html
<select name="experience">
  <option>0-1</option>
  <option>2-4</option>
</select>
```

Playwright:

```python
page.select_option("select[name='experience']", "2-4")
```

---

# 🧠 Step 6 — AI Answering Custom Questions

Sometimes job asks:

> Why do you want to join?

Agent:

1. Reads question text
2. Sends to Gemini
3. Generates answer
4. Fills textarea

Example:

```python
question = page.inner_text(".question-label")

answer = llm.generate(
    f"Answer this job question professionally:\n{question}"
)

page.fill("textarea", answer)
```

---

# 🔍 Step 7 — Portal Detection

Agent checks URL:

```python
url = page.url

if "greenhouse.io" in url:
    handle_greenhouse(page)

elif "lever.co" in url:
    handle_lever(page)

elif "linkedin.com" in url:
    handle_linkedin_easy_apply(page)
```

Each portal has slightly different DOM structure.

Production systems build portal-specific handlers.

---

# ⚠️ Important: How Agent Avoids Breaking

Real issues:

* Dynamic forms
* Multi-step forms
* Required checkboxes
* CAPTCHA
* Login sessions

Production solution:

1. Persist login session
2. Manual CAPTCHA solving
3. SAFE_MODE (review before submit)

---

# 🔄 What Happens If Redirect Happens?

When clicking Apply:

```python
page.click("Apply")

page.wait_for_load_state("networkidle")
```

Agent then:

```python
new_url = page.url
```

And decides flow again.

---

# 🏗️ Real Internal Structure of Apply Tool

```python
def apply_job(job, profile):

    page = open_browser(job["apply_url"])

    fields = detect_fields(page)

    mapping = map_fields(fields, profile)

    fill_fields(page, mapping)

    if safe_mode:
        wait_for_user_confirmation()

    page.click("Submit")

    return status
```

---

# 🚀 So Is It Typing Words?

Yes.

It literally controls a real browser.

Like:

* Moving mouse
* Typing
* Clicking
* Uploading files

Exactly like a human.

---

# 🧠 Advanced Version (AI Field Mapping)

Instead of simple heuristics, advanced systems:

1. Extract full DOM structure
2. Send to LLM:

```text
Here are form fields:
- label: "Given Name"
- id: fname_input

Which profile field should map?
```

LLM returns:

```json
{
  "fname_input": "first_name"
}
```

Then agent fills accordingly.

This is how next-gen automation works.

---

# 🏗️ Production-Grade Apply Tool Components

```text
Apply Engine
 ├── Browser Manager
 ├── Field Detector
 ├── Field Mapper
 ├── Resume Uploader
 ├── Question Answerer (LLM)
 ├── Portal Handler
 └── Submission Validator
```

---

# ⚠️ Critical Reality Check

LinkedIn and many portals detect bots.

They use:

* behavioral detection
* fingerprinting
* rate limiting
* CAPTCHA

So:

* You must throttle actions
* Randomize delays
* Limit applications per day
* Avoid aggressive automation

Otherwise account gets restricted.

---

# 🎯 Final Simplified Explanation

Agent:

1. Opens browser
2. Reads form
3. Matches fields to your data
4. Types like human
5. Uploads resume
6. Clicks submit
7. Records result

Nothing magical.

Just intelligent browser control.

---

# 🧠 Final Analogy

Think of agent as:

A smart intern sitting at your laptop.

You say:

> Apply to this job.

Intern:

* Opens website
* Reads form
* Fills your details
* Uploads resume
* Answers questions
* Submits
* Updates spreadsheet

The only difference?

Your intern is code.

---

If you want next, I can explain:

* How to design a robust Field Detection Engine
* How to build portal-specific handlers
* How to reduce bot detection risk
* Or how to scale this safely to 100+ applications/day

Your move.
